In [ ]:
!pip install pandas numpy openpyxl

In [4]:
from pathlib import Path
from datetime import datetime
import shutil
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"

INPUT_FILENAME = "loan_preprocesado.csv"
OUTPUT_FILENAME = "loan_validado.csv"
LOG_FILENAME = "reporte_errores_validacion.csv"

INPUT_FILE = RAW_DIR / INPUT_FILENAME
OUTPUT_FILE = PROCESSED_DIR / OUTPUT_FILENAME
LOG_FILE = REPORTS_DIR / LOG_FILENAME
README_FILE = PROJECT_ROOT / "README.md"

NORMALIZAR_TEXTO_A_MAYUSCULAS = True

SCORE_MINIMO_ESPERADO = 300
SCORE_MAXIMO_ESPERADO = 850

TASA_MINIMA = 0
TASA_MAXIMA = 100

def preparar_carpetas() -> None:
    """
    Crea las carpetas necesarias para ejecutar el pipeline.
    Esta función no elimina ni reemplaza archivos existentes.
    Solo asegura que las carpetas requeridas estén disponibles.
    """
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    REPORTS_DIR.mkdir(parents=True, exist_ok=True)

preparar_carpetas()

print("Carpetas preparadas:")
print(f"- Entrada:   {RAW_DIR}")
print(f"- Salida:    {PROCESSED_DIR}")
print(f"- Reportes:  {REPORTS_DIR}")

def registrar_log(logs, nivel, etapa, columna, descripcion, filas_afectadas, accion):
    """
    Agrega un registro al listado de logs del proceso.

    Parámetros:
    logs: Lista donde se guardan los eventos de validación.
    nivel: Nivel del evento. Ejemplos: INFO, WARNING, ERROR.
    etapa: Nombre de la etapa donde ocurrió el evento.
    columna: Columna asociada al evento.
    descripcion: Explicación breve del problema o transformación.
    filas_afectadas: Cantidad de filas afectadas por la regla aplicada.
    accion: Acción tomada frente al problema detectado.
    """
    logs.append({
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "nivel": nivel,
        "etapa": etapa,
        "columna": columna,
        "descripcion": descripcion,
        "filas_afectadas": int(filas_afectadas),
        "accion": accion
    })

def cargar_dataset(logs):
    """
    Carga el dataset preprocesado desde la carpeta data/raw/.
    Si el archivo no está en data/raw/ pero sí en la carpeta actual,
    lo copia automáticamente a data/raw/.
    """
    archivo_en_carpeta_actual = PROJECT_ROOT / INPUT_FILENAME

    if not INPUT_FILE.exists() and archivo_en_carpeta_actual.exists():
        shutil.copy(archivo_en_carpeta_actual, INPUT_FILE)
        registrar_log(logs, "INFO", "Carga de datos", "TODAS",
                      "El archivo estaba en la carpeta del notebook y fue copiado a data/raw/.",
                      0, f"Archivo copiado a {INPUT_FILE}")

    if not INPUT_FILE.exists():
        raise FileNotFoundError(
            f"No se encontró {INPUT_FILENAME} en {RAW_DIR}. "
            "Debes ubicarlo en data/raw/ antes de ejecutar."
        )

    df = pd.read_csv(INPUT_FILE)

    registrar_log(logs, "INFO", "Carga de datos", "TODAS",
                  "Dataset preprocesado cargado correctamente desde data/raw/.",
                  len(df), "Lectura de archivo CSV")

    return df

def estandarizar_nombres_columnas(df, logs):
    """
    Estandariza los nombres de las columnas del dataset.

    Reglas aplicadas:
    - Elimina espacios al inicio y final.
    - Convierte a minúsculas.
    - Reemplaza espacios por guiones bajos.
    """
    df = df.copy()
    columnas_originales = df.columns.tolist()

    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
        .str.replace(".", "", regex=False)
    )

    columnas_nuevas = df.columns.tolist()

    registrar_log(logs, "INFO", "Estandarización de columnas", "TODAS",
                  f"Columnas estandarizadas. Antes: {columnas_originales}. Después: {columnas_nuevas}.",
                  0, "Normalización de nombres de columnas")

    return df

def estandarizar_textos(df, logs):
    """
    Limpia columnas de texto.

    Reglas aplicadas:
    - Elimina espacios al inicio y al final.
    - Reemplaza múltiples espacios internos por uno solo.
    - Convierte textos vacíos en valores nulos.
    - Opcionalmente convierte a mayúsculas.
    """
    df = df.copy()
    columnas_texto = df.select_dtypes(include=["object"]).columns.tolist()

    for col in columnas_texto:
        serie_original = df[col].copy()
        df[col] = (df[col].astype(str).str.strip().str.replace(r"\s+", " ", regex=True))
        df[col] = df[col].replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "NaN": pd.NA})

        if NORMALIZAR_TEXTO_A_MAYUSCULAS:
            df[col] = df[col].str.upper()

        cambios = (serie_original.astype(str) != df[col].astype(str)).sum()
        if cambios > 0:
            registrar_log(
                logs=logs,
                nivel="INFO",
                etapa="Estandarización de texto",
                columna=col,
                descripcion="Se limpiaron espacios, valores vacíos y formato textual.",
                filas_afectadas=cambios,
                accion="Normalización de texto"
            )
    return df

def convertir_columnas_numericas(df, logs, umbral_conversion=0.80):
    """
    Intenta convertir a numéricas las columnas de texto que parecen contener números.
    Se aplica solo si al menos el 80% de los valores son convertibles.
    """
    df = df.copy()
    columnas_texto = df.select_dtypes(include=["object"]).columns.tolist()

    for col in columnas_texto:
        serie = df[col].dropna()
        if len(serie) == 0:
            continue
        serie_convertida = pd.to_numeric(serie, errors="coerce")
        proporcion_convertible = serie_convertida.notna().mean()

        if proporcion_convertible >= umbral_conversion:
            nulos_antes = df[col].isna().sum()
            df[col] = pd.to_numeric(df[col], errors="coerce")
            nulos_despues = df[col].isna().sum()

            registrar_log(
                logs=logs,
                nivel="INFO",
                etapa="Conversión numérica",
                columna=col,
                descripcion=f"Columna convertida a numérica. Proporción convertible: {proporcion_convertible:.2f}.",
                filas_afectadas=max(nulos_despues - nulos_antes, 0),
                accion="Conversión con pd.to_numeric(errors='coerce')"
            )
    return df

def estandarizar_formatos(df, logs):
    df = estandarizar_textos(df, logs)
    df = convertir_columnas_numericas(df, logs)
    return df

def eliminar_filas_vacias(df, logs):
    """Elimina filas completamente vacías."""
    df = df.copy()
    filas_antes = len(df)
    df = df.dropna(how="all")
    filas_eliminadas = filas_antes - len(df)
    if filas_eliminadas > 0:
        registrar_log(logs, "WARNING", "Limpieza básica", "TODAS",
                      "Se eliminaron filas completamente vacías.",
                      filas_eliminadas, "dropna(how='all')")
    return df

def eliminar_duplicados(df, logs):
    """Elimina filas duplicadas considerando todas las columnas."""
    df = df.copy()
    filas_antes = len(df)
    df = df.drop_duplicates()
    filas_eliminadas = filas_antes - len(df)
    if filas_eliminadas > 0:
        registrar_log(logs, "WARNING", "Limpieza básica", "TODAS",
                      "Se eliminaron registros duplicados.",
                      filas_eliminadas, "drop_duplicates()")
    return df

def eliminar_registros_con_nulos(df, logs):
    """
    Elimina registros que contienen al menos un valor nulo.
    En un proyecto real algunas columnas podrían imputarse.
    """
    df = df.copy()
    filas_con_nulos = df.isna().any(axis=1)
    cantidad_nulos = filas_con_nulos.sum()
    columnas_con_nulos = df.columns[df.isna().any()].tolist()
    df = df.loc[~filas_con_nulos].copy()
    if cantidad_nulos > 0:
        registrar_log(logs, "WARNING", "Limpieza básica", "TODAS",
                      f"Se eliminaron registros con nulos. Columnas: {columnas_con_nulos}.",
                      cantidad_nulos, "Eliminación de filas con al menos un valor nulo")
    return df

def limpieza_basica(df, logs):
    df = eliminar_filas_vacias(df, logs)
    df = eliminar_duplicados(df, logs)
    df = eliminar_registros_con_nulos(df, logs)
    return df

def obtener_reglas_rango():
    """
    Define reglas de rango para columnas del dataset de préstamos.
    Si una columna no existe en el dataset, simplemente se ignora.
    """
    return {
        "person_age": (18, 100),
        "person_income": (1, 10000000),
        "person_emp_exp": (0, 60),
        "loan_amnt": (1, 1000000),
        "loan_int_rate": (TASA_MINIMA, TASA_MAXIMA),
        "loan_percent_income": (0, 1),
        "cb_person_cred_hist_length": (0, 50),
        "credit_score": (SCORE_MINIMO_ESPERADO, SCORE_MAXIMO_ESPERADO),
        "loan_status": (0, 1)
    }

def validar_rangos(df, logs):
    """
    Elimina filas con valores fuera de rango en columnas definidas.
    """
    df = df.copy()
    reglas = obtener_reglas_rango()

    for col, (valor_min, valor_max) in reglas.items():
        if col not in df.columns:
            continue
        if not pd.api.types.is_numeric_dtype(df[col]):
            continue

        condicion_valida = df[col].between(valor_min, valor_max, inclusive="both")
        fuera_de_rango = (~condicion_valida).sum()
        df = df.loc[condicion_valida].copy()

        if fuera_de_rango > 0:
            registrar_log(logs, "ERROR", "Validación de rangos", col,
                          f"Valores fuera del rango permitido [{valor_min}, {valor_max}].",
                          fuera_de_rango, "Eliminación de registros fuera de rango")
        else:
            registrar_log(logs, "INFO", "Validación de rangos", col,
                          f"OK - Todos los valores en rango [{valor_min}, {valor_max}].",
                          0, "Sin acción requerida")
    return df

def crear_columnas_derivadas(df, logs):
    """
    Crea columnas derivadas relevantes para el análisis de préstamos.

    Columnas posibles:
    - ratio_monto_ingreso: loan_amnt / person_income.
    - categoria_credito: categoría basada en credit_score.
    - fecha_procesamiento: fecha y hora de procesamiento.
    """
    df = df.copy()
    columnas_creadas = []

    if "loan_amnt" in df.columns and "person_income" in df.columns:
        df["ratio_monto_ingreso"] = df["loan_amnt"] / df["person_income"]
        columnas_creadas.append("ratio_monto_ingreso")

    if "credit_score" in df.columns:
        df["categoria_credito"] = pd.cut(
            df["credit_score"],
            bins=[300, 579, 669, 739, 799, 850],
            labels=["Muy malo", "Regular", "Bueno", "Muy bueno", "Excelente"]
        )
        columnas_creadas.append("categoria_credito")

    df["fecha_procesamiento"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    columnas_creadas.append("fecha_procesamiento")

    registrar_log(logs, "INFO", "Columnas derivadas", "TODAS",
                  f"Columnas derivadas creadas: {columnas_creadas}.",
                  len(df), "Creación de nuevas columnas")
    return df

def guardar_resultados(df, logs):
    """
    Guarda el dataset validado y el reporte de validación en CSV.
    """
    df.to_csv(OUTPUT_FILE, index=False)
    df_logs = pd.DataFrame(logs)
    df_logs.to_csv(LOG_FILE, index=False, encoding="utf-8-sig")

    print("Archivos generados correctamente:")
    print(f"- Dataset validado: {OUTPUT_FILE}")
    print(f"- Reporte de validación: {LOG_FILE}")

def actualizar_readme(df_original, df_limpio, logs):
    """
    Actualiza el README.md con un resumen del procesamiento realizado.
    Si el README.md no existe, lo crea automáticamente.
    """
    fecha = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    filas_eliminadas = len(df_original) - len(df_limpio)

    resumen_logs = pd.DataFrame(logs)
    problemas_detectados = 0
    if not resumen_logs.empty:
        problemas_detectados = resumen_logs[
            resumen_logs["nivel"].isin(["WARNING", "ERROR"])
        ]["filas_afectadas"].sum()

    seccion = f'''

## Validación y limpieza del dataset de préstamos

**Fecha de ejecución:** {fecha}

### Archivos utilizados
- Dataset de entrada: `data/raw/loan_preprocesado.csv`
- Dataset validado generado: `data/processed/loan_validado.csv`
- Reporte de validación: `reports/reporte_errores_validacion.csv`

### Resumen del procesamiento
- Filas originales: {len(df_original)}
- Filas finales: {len(df_limpio)}
- Filas eliminadas: {filas_eliminadas}
- Problemas registrados en logs: {int(problemas_detectados)}

### Transformaciones aplicadas
- Estandarización de nombres de columnas.
- Limpieza y normalización de columnas de texto.
- Conversión de columnas numéricas cuando corresponde.
- Eliminación de filas vacías, duplicadas y con nulos.
- Validación de rangos: edad (18-100), credit_score (300-850), tasa interés (0-100).
- Creación de columnas derivadas: ratio_monto_ingreso, categoria_credito.
- Generación de reporte de validación con logs detallados.
'''

    if README_FILE.exists():
        contenido_actual = README_FILE.read_text(encoding="utf-8")
    else:
        contenido_actual = "# Proyecto DataOps - Predicción Incumplimiento Préstamos\n"

    README_FILE.write_text(contenido_actual + seccion, encoding="utf-8")
    print(f"README actualizado en: {README_FILE}")


# ==========================================
# Ejecución del Pipeline
# ==========================================
logs = []

# 1. Cargar datos preprocesados
df_original = cargar_dataset(logs)

# 2. Estandarizar nombres y formatos
df_procesado = estandarizar_nombres_columnas(df_original, logs)
df_procesado = estandarizar_formatos(df_procesado, logs)

# 3. Limpieza básica
df_procesado = limpieza_basica(df_procesado, logs)

# 4. Validar rangos
df_procesado = validar_rangos(df_procesado, logs)

# 5. Crear columnas derivadas
df_procesado = crear_columnas_derivadas(df_procesado, logs)

# 6. Guardar resultados
guardar_resultados(df_procesado, logs)

# 7. Actualizar README
actualizar_readme(df_original, df_procesado, logs)

print("\nResumen del proceso:")
print(f"- Filas originales: {len(df_original)}")
print(f"- Filas finales:    {len(df_procesado)}")
print(f"- Filas eliminadas: {len(df_original) - len(df_procesado)}")
print(f"- Columnas finales: {len(df_procesado.columns)}")

Carpetas preparadas:
- Entrada:   /content/data/raw
- Salida:    /content/data/processed
- Reportes:  /content/reports
Archivos generados correctamente:
- Dataset validado: /content/data/processed/loan_validado.csv
- Reporte de validación: /content/reports/reporte_errores_validacion.csv
README actualizado en: /content/README.md

Resumen del proceso:
- Filas originales: 45000
- Filas finales:    44990
- Filas eliminadas: 10
- Columnas finales: 31
